In [1]:
import pandas as pd

In [2]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [3]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [4]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

law_dense_index = DenseIndex(dense_model, "../data/processed/_dense_law", law_doc)
law_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648
DenseIndex.embeddings:  (176032, 1024)
[dense_index] documents.len: 175933 parent_idx.len: 176032


In [5]:
import text_chunk
from tqdm import tqdm
import json

def get_source(citation):
    if citation in court_consideration_d:
        return court_consideration_d[citation], True
    elif citation in law_d:
        return law_d[citation], False
    else:
        return None, False

train_df = pd.read_csv("../data/train_rewrite_001.csv")

jsonl = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    query = row['query2']
    gold_citation_l = row['gold_citations'].split(';')
    gold_citation_set = set(gold_citation_l)

    for pos_citation in gold_citation_l:
        text, is_court = get_source(pos_citation)
        if text is None:
            continue
            
        d = {}
        d['query'] = query
        
        neg_list = []
        if is_court:
            d['pos'] = [court_consideration_d[pos_citation]]
            court_neg_l = court_dense_index.search_with_score(query, 500)
            for c, score in court_neg_l:
                if c['citation'] in gold_citation_set:
                    continue
                else:
                    neg_list.append(c)
        else:
            d['pos'] = [law_d[pos_citation]]
            law_neg_l = law_dense_index.search_with_score(query, 500)
            for c, score in law_neg_l:
                if c['citation'] in gold_citation_set:
                    continue
                else:
                    neg_list.append(c)

        chunked_neg_list = []
        for neg in neg_list[:5]:
            chunked_neg_list.extend(text_chunk.chunk_with_sliding_window(neg['text'], 384, 128))
            
        d['neg'] = [chunk_neg for chunk_neg in chunked_neg_list]
            
        jsonl.append(d)

with open("../ft_data/train.jsonl", 'w', encoding='utf-8') as f:
    for item in jsonl:
        json_line = json.dumps(item, ensure_ascii=False)
        f.write(json_line + '\n')

100%|██████████| 1139/1139 [05:21<00:00,  3.55it/s]
